# DocStruct — 14-flag ablation sweep on OHR-Bench

Upload this notebook, set **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

Nothing to upload but this file. The corpus fetches itself (`scripts/fetch_ohrbench.py`),
so there is no multi-gigabyte zip in the loop.

**Why OHR-Bench and not our own corpus.** The flags are being measured to decide
defaults that ship. Measuring them against gold we generated ourselves, over documents
we scraped, would let a config win by suiting our own question-writing rather than
retrieval. OHR-Bench gives 3,558 human-authored questions over 95 born-digital
documents that predate this system.

## It is built to be run more than once

Free Colab reclaims sessions without warning. Everything expensive lives on Drive: the
downloaded corpus, the detector-proposal cache, and every finished ablation result.
**If the session dies, reopen and Run all again** — finished flags are restored from
Drive and skipped, and the sweep continues where it stopped.

Budget roughly **6–8 hours** for all 14 flags over 95 documents. Set `N_DOCS` in
section 5 to a smaller number for a faster, lower-power probe.

## 1. GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "NO GPU - Runtime > Change runtime type > T4 GPU, then Run all again."
print('GPU ok:', torch.cuda.get_device_name(0))

# Record the Pillow this kernel has ALREADY loaded, before anything pip-installs over
# it. Colab imports PIL at startup, and PIL._imaging is a compiled C extension: once it
# is in the process it cannot be unloaded, not by deleting sys.modules entries and not
# by reimporting. If pip later upgrades the Python files under it, this kernel keeps the
# old PIL._typing in memory against the new modules, and the mismatch surfaces as
# ImportError: cannot import name '_Ink' from 'PIL._typing' the moment ultralytics loads
# the model. Pinning pillow back to this version after install means the on-disk files
# never actually change, so there is nothing to mismatch.
import PIL
PRELOADED_PILLOW = PIL.__version__
print('kernel loaded pillow', PRELOADED_PILLOW, '- will pin back to this after install')

## 2. Drive — every expensive artefact lives here

`BENCH` is the single directory this notebook owns. Deleting it resets everything;
leaving it alone is what makes a re-run resume.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

BENCH   = '/content/drive/MyDrive/docstruct_bench'
OHRSTORE = BENCH + '/ohrbench_download'   # the 1.5 GB parquet + pdfs.zip, fetched once
CACHE   = BENCH + '/.bench_cache_ohr'     # detector proposals, shared across all 14 runs
REPORTS = BENCH + '/reports/ablations_ohr'
for d in (BENCH, OHRSTORE, CACHE, REPORTS):
    os.makedirs(d, exist_ok=True)

# Optional: the vision detector's weights. The sweep runs without them (geometry-only).
# To use the hybrid path instead, put yolov8m-doclaynet.pt (52 MB) in Drive at this path.
WEIGHTS_DRIVE = BENCH + '/yolov8m-doclaynet.pt'

print('bench dir ->', BENCH)
print('weights   ->', 'found' if os.path.exists(WEIGHTS_DRIVE) else 'absent (will run geometry-only)')

## 3. Repo

In [ ]:
%cd /content
if not os.path.exists('/content/DocStruct/.git'):
    !git clone -q -b feat/paper-draft https://github.com/CandyButcher27/DocStruct
%cd /content/DocStruct
!git pull -q --ff-only 2>/dev/null || echo '(could not fast-forward; keeping what is here)'
!git log --oneline -1

## 4. Install

`retrieval` (embeddings), `benchmark` (comparison adapters — ablate.py scores one tool,
but importing the eval package touches all of them), `model` (YOLO detector), and
`pyarrow`, which `fetch_ohrbench.py` needs to read the parquet and is not a core
dependency.

**Ignore pip's dependency-conflict warnings** about `jedi`, `requests`/`google-colab`
and `opentelemetry`: pip reports on the whole environment, including Colab preinstalls
this run never imports. Only a traceback matters.

In [ ]:
!pip install -q -e ".[retrieval,benchmark,model]"
!pip install -q pyarrow

# Put Pillow back to the version this kernel loaded at startup (see section 1). The
# alternative -- upgrading and restarting the runtime -- cannot be done inside Run all.
!pip install -q "pillow=={PRELOADED_PILLOW}"

import PIL
from PIL import Image
Image.new('RGB', (4, 4))
assert PIL.__version__ == PRELOADED_PILLOW, (
    f'pillow is {PIL.__version__}, kernel loaded {PRELOADED_PILLOW} -- the C extension and the '
    'Python files disagree. Restart the runtime and Run all again.'
)
print('pillow', PIL.__version__, 'ok (core and files agree)')

In [ ]:
# Ultralytics ships a top-level `tests` package that shadows ours, and importing YOLO is
# what puts it on the path. Confirm the real PIL survives an ultralytics import before the
# long run rather than partway through the first ablation.
from ultralytics import YOLO
from PIL import Image
Image.new('RGB', (4, 4))
print('ultralytics imported, PIL still alive')

# get_adapters() swallows import errors and drops anything whose available() is False,
# so a missing dependency would silently shrink the run instead of failing loudly.
from docstruct.eval.adapters import get_adapters
assert 'docstruct' in get_adapters(names=['docstruct'], weights=None), \
    'docstruct adapter did not import - check the traceback above'
print('docstruct adapter: ok')

## 5. Corpus — fetched once, cached on Drive

`fetch_ohrbench.py` downloads OHR-Bench's parquet and PDF zip (~1.5 GB), then keeps
only documents that have questions, are born-digital, and live in a multi-page domain.
The download is symlinked to Drive so a later session re-uses it instead of pulling
1.5 GB again. Expect ~95 PDFs and ~3,558 gold rows.

In [ ]:
N_DOCS = 0      # 0 = all 95 documents. Set e.g. 30 for a faster, lower-power probe.

# Point the fetcher's download cache at Drive before it runs, so the 1.5 GB arrives once.
os.makedirs('data/ohrbench', exist_ok=True)
if not os.path.islink('data/ohrbench/_download'):
    if os.path.isdir('data/ohrbench/_download'):
        !cp -rn data/ohrbench/_download/* "{OHRSTORE}/" 2>/dev/null
        !rm -rf data/ohrbench/_download
    os.symlink(OHRSTORE, 'data/ohrbench/_download')
print('download cache ->', os.path.realpath('data/ohrbench/_download'))

!python scripts/fetch_ohrbench.py

import glob, json
n_pdfs = len(glob.glob('data/ohrbench/*.pdf'))
gold = json.load(open('data/qa/ohrbench.json', encoding='utf-8'))
print(f'\nPDFs: {n_pdfs}   gold rows: {len(gold)}')
assert n_pdfs >= 90, f'only {n_pdfs} PDFs - the fetch did not complete'
assert len(gold) > 3000, f'only {len(gold)} gold rows - the fetch did not complete'

## 6. Which tool the sweep measures

With the weights present this measures the shipped hybrid path. Without them it
measures geometry-only, which is a legitimate configuration — the paper reports the
vision detector as null on every corpus it could be checked against — but the two are
not comparable to each other, so do not mix results from both into one table.

In [ ]:
if os.path.exists(WEIGHTS_DRIVE):
    os.makedirs('weights', exist_ok=True)
    !cp -n "{WEIGHTS_DRIVE}" weights/yolov8m-doclaynet.pt
    TOOL, WEIGHTS = 'docstruct', 'weights/yolov8m-doclaynet.pt'
else:
    TOOL, WEIGHTS = 'docstruct_geo', None

QA, PDFS = 'data/qa/ohrbench.json', 'data/ohrbench'
print('tool   :', TOOL)
print('weights:', WEIGHTS or '(none - geometry-only)')

## 7. Smoke — before scaling

Per project convention: never start the sweep before one run works end to end on the
exact path the sweep uses. Read the numbers, not just the exit code — `benchmark_tool`
catches per-document exceptions internally, so a config that breaks chunking on every
document can still exit 0 with an all-errors row, and gold that does not match the
PDFs scores zero while still looking healthy.

In [ ]:
import subprocess, sys, json

args = [sys.executable, 'scripts/ablate.py', '--name', 'ab_smoketest', '--tool', TOOL,
        '--pdfs-dir', PDFS, '--qa', QA, '--cache-dir', CACHE, '--limit-docs', '2']
if WEIGHTS:
    args += ['--weights', WEIGHTS]
print(subprocess.run(args, capture_output=True, text=True).stdout[-1500:])

# ablate.py nests every metric under "metrics"; the top level holds name/config/per_doc.
m = json.load(open('reports/ablations/ab_smoketest.json'))['metrics']
print('MRR:', m['mrr'], '| questions:', m['n_questions'], '| errors:', m['errors'])
assert m['errors'] == 0, f"{m['errors']} errors on the smoke docs - fix before the sweep"
assert m['n_questions'] > 0, '0 questions scored - gold does not match these PDFs'
assert m['mrr'] > 0, 'MRR is 0 on the smoke docs - something is broken, not just unlucky'
print('smoke test ok')

## 8. The sweep — 14 flags, resumable

Every gated feature in `config.py` ships default-OFF because nothing measured it. This
is that measurement.

Each result is copied to Drive the moment it finishes, not at the end, so a reclaimed
session loses at most the flag in flight. Re-running this cell restores any flag already
on Drive and skips it. The model-proposal cache is shared and config-independent, so
only the first run anywhere pays full detector cost; geometry proposals are keyed on the
config fingerprint and are correctly recomputed per flag.

In [ ]:
import os, shutil, subprocess, sys, json, time

RUNS = [
    ("baseline", None),
    ("dedupe", "DEDUPE_CHARS=True"),
    ("dehyphen", "DEHYPHENATE=True"),
    ("normalize", "NORMALIZE_TEXT=True"),
    ("fig_area", "FIGURE_OVERLAP_BY_AREA=True"),
    ("multicol", "MULTI_COLUMN=True"),
    ("bandsplit", "BAND_SPLIT=True"),
    ("furniture", "STRIP_PAGE_FURNITURE=True"),
    ("tbl_borderless", "TABLE_TEXT_STRATEGY_FALLBACK=True"),
    ("tbl_keyvalue", "TABLE_SERIALIZATION=keyvalue"),
    ("tbl_split", "TABLE_SPLIT_ROWS=True"),
    ("hdr_bold", "HEADER_RANK_BY_WEIGHT=True"),
    ("keep_refs", "KEEP_REFERENCES=True"),
    ("label_contain", "LABEL_AWARE_CONTAINMENT=True"),
]

os.makedirs('reports/ablations', exist_ok=True)
for flag, override in RUNS:
    name = f"ab_{flag}"
    drive_json, local_json = f"{REPORTS}/{name}.json", f"reports/ablations/{name}.json"

    if os.path.exists(drive_json):
        print(f"== {name}: already on Drive, restoring and skipping", flush=True)
        shutil.copy(drive_json, local_json)
        continue

    print(f"== {name}: running ({override or 'no override'})", flush=True)
    t0 = time.time()
    args = [sys.executable, "scripts/ablate.py", "--name", name, "--tool", TOOL,
            "--pdfs-dir", PDFS, "--qa", QA, "--cache-dir", CACHE]
    if WEIGHTS:
        args += ["--weights", WEIGHTS]
    if N_DOCS:
        args += ["--limit-docs", str(N_DOCS)]
    if override:
        args += ["--set", override]
    proc = subprocess.run(args, capture_output=True, text=True)
    print("\n".join(proc.stdout.splitlines()[-8:]))
    if proc.returncode != 0:
        print(proc.stderr[-2000:])
        raise SystemExit(f"{name} failed - fix before continuing the sweep")

    shutil.copy(local_json, drive_json)
    print(f"   -> saved to Drive ({time.time() - t0:.0f}s)", flush=True)

print("\n=== SWEEP DONE ===")

## 9. Compare every flag against baseline

In [ ]:
import json, glob

def metrics(path):
    d = json.load(open(path))
    return d['name'], d['metrics']          # metrics are nested, not top-level

_, base = metrics('reports/ablations/ab_baseline.json')
base_mrr = base['mrr']

rows = []
for f in sorted(glob.glob('reports/ablations/ab_*.json')):
    name, m = metrics(f)
    if name in ('ab_baseline', 'ab_smoketest'):
        continue
    rows.append((name, m['mrr'], m['mrr'] - base_mrr, m['ndcg'], m['recall'], m['errors']))
rows.sort(key=lambda r: -r[2])

print(f"tool={TOOL}  docs={'all' if not N_DOCS else N_DOCS}  questions={base['n_questions']}")
print(f"{'flag':22} {'MRR':>8} {'delta':>8} {'NDCG':>8} {'Recall':>8} {'errors':>7}")
print(f"{'baseline':22} {base_mrr:8.4f} {'--':>8} {base['ndcg']:8.4f} {base['recall']:8.4f} {base['errors']:7}")
for name, mrr, delta, ndcg, recall, err in rows:
    flag = 'WIN' if delta > 0.001 else ('LOSS' if delta < -0.001 else 'flat')
    print(f"{name:22} {mrr:8.4f} {delta:+8.4f} {ndcg:8.4f} {recall:8.4f} {err:7}  {flag}")

A delta is not a result on its own — a flag that moves MRR by less than the
document-to-document spread has not been shown to do anything. Treat `WIN` as a
candidate to flip in `config.py`, with this run named in the comment as its
justification (project rule 5), and prefer flags whose gain survives a paired test.

## 10. Collect

In [ ]:
!cd reports && zip -q -r /content/docstruct_ablations_ohr.zip ablations/*.json
print('results are on Drive at', REPORTS)
try:
    from google.colab import files
    files.download('/content/docstruct_ablations_ohr.zip')
except Exception as e:
    print('browser download skipped:', e)